In [ ]:
import os
import sys
import csv
from pathlib import Path


# --- OpenStudio Path Setup ---
OPENSTUDIO_VERSION = "3.11.0"
openstudio_path = f"/Applications/OpenStudio-{OPENSTUDIO_VERSION}/Python"

if Path(openstudio_path).exists():
    sys.path.insert(0, openstudio_path)
import openstudio

def get_prop_value(props, name):
    """Helper to safely extract value from AdditionalProperties by type."""
    if props.getFeatureAsDouble(name).is_initialized():
        return props.getFeatureAsDouble(name).get()
    if props.getFeatureAsString(name).is_initialized():
        return props.getFeatureAsString(name).get()
    if props.getFeatureAsInteger(name).is_initialized():
        return props.getFeatureAsInteger(name).get()
    return None

def extract_scenario_data(osm_path, scenario_name):
    """
    Loads an OSM and extracts all target properties from any 
    AdditionalProperties object in the model.
    """
    results = {"scenario": scenario_name}
    
    vt = openstudio.osversion.VersionTranslator()
    model_ptr = vt.loadModel(openstudio.toPath(str(osm_path)))
    
    if not model_ptr.is_initialized():
        print(f"  ✗ Failed to load: {osm_path.name}")
        return None

    model = model_ptr.get()
    
    # Get all objects of type OS:AdditionalProperties
    idd_type = openstudio.IddObjectType("OS:AdditionalProperties")
    all_props_objects = model.getObjectsByType(idd_type)
    
    found_any = False
    for obj in all_props_objects:
        # CORRECT CASTING: Use toAdditionalProperties to access feature methods
        opt_props = openstudio.model.toAdditionalProperties(obj)
        
        if opt_props.is_initialized():
            props = opt_props.get()
            feature_names = props.featureNames()
            
            # 1. Handle Embodied Carbon (Requires 'name' prefix)
            if "total_additional_embodied_carbon_kgCO2" in feature_names:
                measure_name = get_prop_value(props, "name")
                val = get_prop_value(props, "total_additional_embodied_carbon_kgCO2")
                if measure_name and val is not None:
                    header = f"{measure_name} total_additional_embodied_carbon_kgCO2"
                    results[header] = val
                    found_any = True

            # 2. Handle Operating Data
            operating_keys = [
                "annual_electricity_cost_usd",
                "annual_gas_cost_usd",
                "annual_electricity_operating_emissions_kg_co2e",
                "annual_gas_operating_emissions_kg_co2e"
            ]
            for key in operating_keys:
                if key in feature_names:
                    val = get_prop_value(props, key)
                    if val is not None:
                        results[key] = val
                        found_any = True
                
    return results if found_any else None

def main(root_folder):
    root_path = Path(root_folder)
    all_data = []
    all_headers = set()
    
    print("="*80)
    print(f"GENERATING PARAMETRIC RECAP FROM: {root_path}")
    print("="*80)

    # Find relevant OSM files
    osm_files = [p for p in root_path.rglob("*.osm") if p.name in ["in.osm", "in_modified.osm"]]

    for osm_path in sorted(osm_files):
        # Improved scenario name logic
        parts = list(osm_path.parts)
        try:
            # Look for the index of 'run' and take the folder before it
            idx = parts.index('run')
            scenario = parts[idx-1]
        except ValueError:
            # Fallback: if 'run' isn't in path, use the folder containing the file
            scenario = osm_path.parent.name
            
        print(f"Processing Scenario: {scenario}")
        
        data = extract_scenario_data(osm_path, scenario)
        if data:
            all_data.append(data)
            all_headers.update(data.keys())

    if not all_data:
        print("\n✗ No matching data found in the AdditionalProperties blocks.")
        return

    # Define Header Order
    fixed_headers = [
        "scenario",
        "annual_electricity_cost_usd",
        "annual_gas_cost_usd",
        "annual_electricity_operating_emissions_kg_co2e",
        "annual_gas_operating_emissions_kg_co2e"
    ]
    
    # Identify measure-specific embodied headers
    embodied_headers = sorted([h for h in all_headers if h not in fixed_headers and h != "scenario"])
    fieldnames = fixed_headers + embodied_headers
    
    csv_path = root_path / "parametric_results.csv"
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(all_data)

    print("\n" + "="*80)
    print(f"COMPLETE: {len(all_data)} scenarios successfully processed.")
    print(f"Report saved to: {csv_path}")
    print("="*80)

if __name__ == "__main__":
    TARGET_DIRECTORY = "./simulations/run_test_004" 
    main(TARGET_DIRECTORY)

GENERATING PARAMETRIC RECAP FROM: simulations/run_test_004
Processing Scenario: baseline_SmallOffice_Amarillo
Processing Scenario: baseline_SmallOffice_Amarillo
Processing Scenario: baseline_SmallOffice_Amarillo
Processing Scenario: _proto
Processing Scenario: _proto
Processing Scenario: _proto
Processing Scenario: door_wooden_d_SmallOffice_Amarillo
Processing Scenario: _proto
Processing Scenario: _proto
Processing Scenario: _proto
Processing Scenario: roof_r40_SmallOffice_Amarillo
Processing Scenario: _proto
Processing Scenario: _proto
Processing Scenario: _proto
Processing Scenario: wall_r30_SmallOffice_Amarillo
Processing Scenario: _proto
Processing Scenario: _proto
Processing Scenario: _proto
Processing Scenario: window_u0.2_SmallOffice_Amarillo

COMPLETE: 5 scenarios successfully processed.
Report saved to: simulations/run_test_004/parametric_results.csv
